<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Structured_Outputs_Project_Log_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2: Log Analysis and Structured Incident Reports

Companion notebook for the lesson **Applied Structured Outputs: Three Mini Projects**.

We turn raw application logs into a typed incident report an alerting stack can consume. Logs bring two problems that plain ticket text did not have: they contain secrets, and most of their bulk is noise. So the pipeline has three stages: **redact**, **aggregate**, **report**.

**What you will build:**

1. A redaction function that strips secrets before any text reaches an external API.
2. A plain-Python aggregation step that hands the model computed statistics instead of asking it to count.
3. A structured `IncidentReport` generated by the model and re-validated with Pydantic.
4. A confidence gate that routes uncertain reports to a human before anyone gets paged.

## Install Packages and Set Up the Provider

Pick your provider by setting `PROVIDER` below. Gemini is the course default and its free tier covers this notebook.

> **Colab users:** store your API key in **Secrets** (the key icon in the left sidebar) under the name shown for your provider (`GOOGLE_API_KEY`, `OPENAI_API_KEY`, or `ANTHROPIC_API_KEY`). The cell falls back to an interactive prompt if no secret is found.

In [1]:
# Shared install profile for the structured outputs project notebooks
!pip install -q google-genai==2.16.0 openai==2.51.0 anthropic==0.120.2 pydantic==2.13.4 pandas==3.0.5 tqdm==4.70.0

In [2]:
import os
import getpass

PROVIDER = "gemini"  # "gemini" | "openai" | "anthropic"

KEY_ENV = {
    "gemini": "GOOGLE_API_KEY",
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
}
env_var = KEY_ENV[PROVIDER]

# Option 1: Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ[env_var] = userdata.get(env_var)
except Exception:
    pass

# Option 2: interactive prompt (fallback)
if not os.getenv(env_var):
    os.environ[env_var] = getpass.getpass(f"Enter {env_var}: ")

print(f"[OK] {env_var} is set")

[OK] GOOGLE_API_KEY is set


### The `extract()` Helper

All project code goes through one function, `extract(prompt, schema)`. It sends a prompt and returns a validated Pydantic object, using the native structured output API of whichever provider you selected. We use each provider's small, fast model: extraction is high-volume, low-difficulty work, and the flagship models cost several times more per token while adding little on tasks this constrained. (Model IDs current as of July 2026; swap in the provider's latest small model if these have been superseded.)

In [3]:
from pydantic import BaseModel

MODELS = {
    "gemini": "gemini-3.5-flash-lite",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-haiku-4-5",
}

if PROVIDER == "gemini":
    from google import genai
    client = genai.Client()
elif PROVIDER == "openai":
    from openai import OpenAI
    client = OpenAI()
elif PROVIDER == "anthropic":
    import anthropic
    client = anthropic.Anthropic()

def extract(prompt: str, schema: type[BaseModel]) -> BaseModel:
    """Send a prompt and return a validated instance of `schema`."""
    if PROVIDER == "gemini":
        response = client.models.generate_content(
            model=MODELS["gemini"],
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "response_schema": schema,
            },
        )
        if response.parsed is None:
            raise ValueError("Model returned no parseable output")
        return response.parsed
    if PROVIDER == "openai":
        response = client.responses.parse(
            model=MODELS["openai"],
            input=prompt,
            text_format=schema,
            reasoning={"effort": "minimal"},
        )
        return response.output_parsed
    if PROVIDER == "anthropic":
        response = client.messages.parse(
            model=MODELS["anthropic"],
            max_tokens=2048,
            messages=[{"role": "user", "content": prompt}],
            output_format=schema,
        )
        return response.parsed_output
    raise ValueError(f"Unknown provider: {PROVIDER}")

print(f"[OK] extract() ready, provider={PROVIDER}, model={MODELS[PROVIDER]}")

[OK] extract() ready, provider=gemini, model=gemini-3.5-flash-lite


## The Synthetic Log Dataset

This excerpt simulates an API gateway during a bad deploy. It includes normal requests, a burst of 500 errors, a database timeout, and (deliberately) a leaked API token on the `WARN` line.

In [4]:
raw_logs = """
2026-07-10T12:00:01Z INFO  api-gw request_id=aa1 path=/courses/search status=200 latency_ms=120
2026-07-10T12:00:05Z ERROR api-gw request_id=aa2 path=/courses/search status=500 latency_ms=5100 err="ValueError: invalid param: dateRange"
2026-07-10T12:00:06Z ERROR api-gw request_id=aa3 path=/courses/search status=500 latency_ms=4980 err="ValueError: invalid param: dateRange"
2026-07-10T12:00:07Z WARN  auth token=sk_live_ABC123XYZ leaked_in_log=true user=jdoe
2026-07-10T12:00:08Z ERROR db  query="SELECT ..." status=timeout duration_ms=30000
2026-07-10T12:00:09Z ERROR api-gw request_id=aa4 path=/courses/search status=500 latency_ms=5200 err="ValueError: invalid param: dateRange"
2026-07-10T12:02:10Z INFO  api-gw request_id=aa5 path=/home status=200 latency_ms=80
""".strip()

print(raw_logs.splitlines()[0])
print("lines:", len(raw_logs.splitlines()))

2026-07-10T12:00:01Z INFO  api-gw request_id=aa1 path=/courses/search status=200 latency_ms=120
lines: 7


## Step 1: Redact Secrets Before the LLM

Logs routinely capture API keys, session tokens, and personal data. Sending them to an external API copies them into another company's infrastructure, expands your audit surface, and can violate data-minimization requirements under frameworks like GDPR and SOC 2.

The verification step matters as much as the regex: a redaction function that silently stops matching is worse than none, because you believe you are protected. Keep assertions like these in the pipeline itself.

In [5]:
import re

def redact(text: str) -> str:
    """Replace sensitive values with placeholders before LLM processing."""
    text = re.sub(r"sk_live_[A-Za-z0-9]+", "sk_live_REDACTED", text)
    text = re.sub(r"(token=)(\S+)", r"\1REDACTED", text)
    return text

safe_logs = redact(raw_logs)
print("Contains raw token:", "sk_live_ABC" in safe_logs)
print("Contains REDACTED: ", "REDACTED" in safe_logs)
assert "sk_live_ABC" not in safe_logs

Contains raw token: False
Contains REDACTED:  True


> **Security note:** a regex denylist only catches patterns you anticipated. For higher-stakes pipelines, invert the approach: parse the logs into fields and pass an explicit allowlist of safe fields to the model, dropping everything else by default. The exercises walk you through both upgrades.

## Step 2: Aggregate Statistics in Plain Python

We could dump the redacted logs into the prompt and ask for a report. That works at seven lines and degrades at seven million. Counting error rates and grouping failure signatures is deterministic work that Python does exactly and for free, so we do it before the model call and hand the model the totals. The model then interprets numbers instead of counting them, which LLMs do unreliably.

In [6]:
from collections import Counter

def aggregate_logs(log_text: str) -> dict:
    lines = log_text.splitlines()
    status_5xx, err_counter, path_5xx = 0, Counter(), Counter()

    for line in lines:
        status = re.search(r"status=(\d+)", line)
        path = re.search(r"path=(\S+)", line)
        err = re.search(r'err="([^"]+)"', line)
        if status and int(status.group(1)) >= 500:
            status_5xx += 1
            path_5xx[path.group(1) if path else "unknown"] += 1
            if err:
                err_counter[err.group(1)] += 1

    return {
        "total_lines": len(lines),
        "errors_5xx": status_5xx,
        "top_error_signatures": err_counter.most_common(5),
        "top_error_paths": path_5xx.most_common(5),
    }

stats = aggregate_logs(safe_logs)
stats

{'total_lines': 7,
 'errors_5xx': 3,
 'top_error_signatures': [('ValueError: invalid param: dateRange', 3)],
 'top_error_paths': [('/courses/search', 3)]}

## Step 3: Generate the Incident Report

The report schema mirrors what an SRE would write by hand: impact, suspected cause, evidence, recommended actions, plus an escalation flag your paging system can act on.

In [7]:
from pydantic import Field
from typing import List

class IncidentReport(BaseModel):
    incident_title: str = Field(description="Short title")
    impact: str = Field(description="Who or what is affected, and how badly")
    suspected_root_cause: str = Field(description="Best-guess root cause")
    evidence: List[str] = Field(description="Evidence drawn from the logs and stats")
    recommended_actions: List[str] = Field(description="Concrete next steps for engineers")
    needs_escalation: bool = Field(description="True if on-call escalation is needed")
    confidence: float = Field(ge=0, le=1, description="Self-reported confidence, 0-1")

prompt = f"""You are an SRE assistant. Given the log excerpt and precomputed stats,
write an incident report for engineers.

LOGS:
{safe_logs}

STATS:
{stats}
"""

report = extract(prompt, IncidentReport)
print("[OK] report generated")

[OK] report generated


In [8]:
import json

print(json.dumps(report.model_dump(), indent=2))

{
  "incident_title": "API Gateway 500 Errors on Course Search Due to Invalid dateRange Parameter",
  "impact": "Users attempting to search courses via /courses/search are experiencing 500 Internal Server Errors and high latency.",
  "suspected_root_cause": "Malformed dateRange parameter values passed to the course search endpoint are triggering unhandled ValueError exceptions in the application logic.",
  "evidence": [
    "3 errors with status 500 logged on path /courses/search",
    "Error signature: ValueError: invalid param: dateRange",
    "Latencies for failed requests exceeded 4900ms",
    "Database timeout observed during the incident timeframe"
  ],
  "recommended_actions": [
    "Investigate the client-side payloads sending invalid dateRange parameters to /courses/search",
    "Implement robust input validation and sanitization for dateRange parameters in the API gateway or backend service",
    "Check database performance and query optimization for course search"
  ],
  "ne

Look at the evidence list: a good run flags the leaked token for rotation even though we redacted its value. That is the property good redaction aims for: remove the value, keep the signal.

## Step 4: The Confidence Gate

Same pattern as Project 1: the pipeline acts on confident reports and queues uncertain ones for a person. Here the action is paging someone at 3am, so the gate earns its keep.

In [9]:
CONFIDENCE_THRESHOLD = 0.7

def route_report(report: IncidentReport) -> str:
    if report.confidence < CONFIDENCE_THRESHOLD:
        return "human-review-queue"
    if report.needs_escalation:
        return "page-on-call"
    return "log-only"

print("Routing decision:", route_report(report))

Routing decision: page-on-call


## Exercises

1. **More redaction patterns.** Extend `redact()` to cover email addresses, IPv4 addresses, and `Bearer` tokens. Write the assertions first, then the regexes.
2. **The allowlist version.** Parse each log line into fields and rebuild the pipeline so only an explicit allowlist of fields ever reaches the prompt.
3. **Chunk and merge.** Split a larger log file into chunks, generate one report per chunk, then merge them: union the evidence and actions, OR the escalation flags, and average confidence. When does merging produce a worse report than one big prompt?
4. **Wire the gate.** Collect reports routed to `human-review-queue` in a list and render them as a review table with `pandas`.